# TypeSafe 快速开始实验（Quickstart Lab）

针对官方文档对应章节的可运行实验笔记，全部实验使用**中文场景与中文提示词**。
面向会基础 Python、刚接触 AI Agent 的读者。

**学习目标：** 把官方 Stripe 连接故障场景从 Playground 迁移到 HTTP 和 Python SDK，并正确读取三种答案。

[官方原文](https://docs.typesafe.ai/introduction/quickstart) · [中文参考](https://bald0wang.github.io/jev-docs-zh/introduction/quickstart/)。本章以中文重述理论、复刻对应场景；扩展实验会单独说明。
所有客户、订单及消息均为教学合成数据。

## 笔记本结构

| 章节 | 内容 |
|---|---|
| 0. 准备 | 安装库、配置客户端、连通性测试与离线示例 |
| 1 | Playground 与最小 Noul |
| 2 | HTTP 请求体与环境变量 |
| 3 | Python 混合三个原语 |
| 练习与小结 | 练习、自查、总结与本次执行记录 |

实验按**原理 → 理论根基 → 定义数据 → 定义问题 → 调用 → 解读结果**展开，每个代码单元格只做一件事。

## 运行要求

- Python ≥ 3.10；本章使用 `typesafe-sdk==0.7.0`。
- 真实实验需要启动进程的 `TYPESAFE_API_KEY` 环境变量，密钥不要写进 Notebook。

在本仓库 `notebooks/` 目录创建环境并打开本文件：

```bash
./setup_env.sh
.venv/bin/python -m pip install -r requirements.txt -c generators/constraints-foundations.txt
.venv/bin/jupyter lab quickstart_experiments.ipynb
```

产品名、字段名和选项 key 保持英文，state、提示词与解说使用中文。
默认 `JEV_RUN_MODE=live`，调用失败即停止；无密钥学习时，在启动 Jupyter 前设置 `JEV_RUN_MODE=offline`。
`auto` 仅供教学体验，缺密钥或 401 时显式回退；正式验收使用 `live`。

**验证状态：真实 API 待验收。** 本文件尚未执行真实 API；离线检查仅验证代码路径。
批量执行、离线预览和验收记录见本目录 `MAINTENANCE.md`。

## 0. 准备

本节可折叠阅读，但独立运行时不能跳过。客户端、辅助对象和示例数据都在本文件中定义。

### 0.1 安装依赖

推荐先运行 `setup_env.sh`。只有当前内核缺少 SDK 时，本格才安装依赖。

In [ ]:
import importlib.util
if importlib.util.find_spec("typesafe_sdk") is None:
    %pip install -q typesafe-sdk==0.7.0

**观察与理解：** 安装包的名字是 typesafe-sdk，Python 导入名是 typesafe_sdk。安装成功不代表 API 已连通。

### 0.2 导入与配置

默认模型固定版本，便于记录实验条件；可通过环境变量更换。不要从 Notebook 输入密钥。

In [ ]:
import os
import json
import time
import math
from datetime import datetime, timezone
from importlib.metadata import version
from typesafe_sdk import (
    Choice, Score, Noul, NoulCriteria, TypeSafeClient,
    TypeSafeAuthenticationError, RetryPolicy,
)

MODEL = os.environ.get("TYPESAFE_DEFAULT_MODEL", "jev-1.13.0")
RUN_MODE = os.environ.get("JEV_RUN_MODE", "live")
API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
if RUN_MODE not in {"live", "offline", "auto"}:
    raise ValueError("JEV_RUN_MODE 只能是 live、offline 或 auto")
if RUN_MODE == "live" and not API_KEY:
    raise RuntimeError("请在启动 Jupyter 前配置 TYPESAFE_API_KEY 环境变量")
client = None
if RUN_MODE != "offline" and API_KEY:
    client = TypeSafeClient(api_key=API_KEY, model=MODEL, timeout=30,
                           retry=RetryPolicy(max_retries=0))
print("模式：", RUN_MODE, "SDK：", version("typesafe-sdk"), "模型配置：", MODEL)

正式验收禁用自动回退，且不自动重试，以便请求数量有界。`auto` 与 `offline` 是教学工具，不代表成功连接模型。

### 0.3 连通性测试

用一条 Noul 检查真实响应能否返回。网络、限流与输入错误直接抛出，不伪装成不确定判断。

In [ ]:
PING = {"source": "offline", "reason": "未发起连通性请求"}
if client is not None:
    try:
        ping = client.system_one("你好", {"greeting": Noul(
            instructions="这段文字是否在打招呼？")})
        PING = {"source": "live", "model": ping.model,
                "input_tokens": ping.usage.input_tokens,
                "output_tokens": ping.usage.output_tokens}
    except TypeSafeAuthenticationError:
        if RUN_MODE == "live":
            raise
        client.close()
        client = None
        PING["reason"] = "401 鉴权失败，仅教学模式允许回退"
print(json.dumps(PING, ensure_ascii=False))

**观察与理解：** source=live 表示这一次连通性请求成功；仍要查看后续实验记录，不能用它代替整章验收。

### 0.4 离线替身

沿用参考模板的 `_FakeAnswer` 与 `_FakeResponse` 访问方式。人工数字仅用来检验读取字段和代码分支。

In [ ]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for name, value in values.items():
            setattr(self, name, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.model = "人工示例，非模型预测"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)

人工 Score 由概率计算期望，避免模板中的分数与分布不一致。人工 confidence 只是指定的演示字段，不是在复现服务端的计算公式。

定义两种示例答案构造器；Noul 可直接用 `_FakeAnswer`。所有具体答案集中在下一节。

In [ ]:
def fake_choice(probabilities, confidence):
    return _FakeAnswer("choice", choice=max(probabilities, key=probabilities.get),
                       probabilities=probabilities, confidence=confidence)


def fake_score(probabilities, legend, confidence):
    return _FakeAnswer("score", score=sum(k * p for k, p in probabilities.items()),
                       probabilities=probabilities, legend=dict(enumerate(legend)),
                       confidence=confidence)

**观察与理解：** 例如概率 {0:0.05, 1:0.26, 2:0.69} 对应 1.64。不能把另一个数与该分布配在一起。

### 0.5 统一调用入口

每次调用记录来源、模型与 token 用量。离线耗时记为 None，不把本地字典访问当成模型速度。

In [ ]:
CALL_LOG = []


class TS:
    def call(self, state, questions, offline_answers, label):
        start = time.perf_counter()
        source = "live"
        if client is None:
            response, source = _FakeResponse(offline_answers), "offline"
        else:
            try:
                response = client.system_one(state, questions)
            except TypeSafeAuthenticationError:
                if RUN_MODE != "auto":
                    raise
                response, source = _FakeResponse(offline_answers), "offline"
        validate_response(response, questions)
        CALL_LOG.append({"case": label, "source": source, "model": response.model,
                         "seconds": time.perf_counter() - start if source == "live" else None,
                         "input_tokens": response.usage.input_tokens,
                         "output_tokens": response.usage.output_tokens})
        if source == "offline":
            print("离线示例：", label, "；人工答案，不是 Jev 实测")
        return response


ts = TS()

与参考模板相比，这里增加了严格 live 模式和逐次记录。保留 401 教学回退，但超时、429 等错误继续失败，防止验收被回退掩盖。

校验结构与数值契约；只断言接口应满足的性质，不断言真实模型必须预测某个标签。

In [ ]:
def validate_response(response, questions):
    if set(response.answers) != set(questions):
        raise ValueError("答案 ID 与问题 ID 不一致")
    for key, question in questions.items():
        answer = response.answers[key]
        if isinstance(question, Noul):
            if not 0 <= answer.noul <= 1:
                raise ValueError("Noul 超出概率范围")
            continue
        probabilities = answer.probabilities
        if not all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities.values()):
            raise ValueError("概率值无效")
        if not math.isclose(sum(probabilities.values()), 1, abs_tol=0.02):
            raise ValueError("概率之和偏离 1")
        if not 0 <= answer.confidence <= 1:
            raise ValueError("confidence 超出范围")
        if isinstance(question, Choice):
            if set(probabilities) != set(question.criteria):
                raise ValueError("Choice 选项集合不一致")
            if answer.choice not in probabilities:
                raise ValueError("Choice 标签不在选项中")
        else:
            expected = sum(int(k) * p for k, p in probabilities.items())
            if not math.isclose(answer.score, expected, abs_tol=0.03):
                raise ValueError("Score 与概率加权期望不一致")

**观察与理解：** 容差用于服务端数值舍入。结构检查通过只说明响应可读取，不证明语义判断正确。

显示结果时统一列出类型、概率和置信度；Noul 不额外制造 confidence 字段。

In [ ]:
def show(response):
    rows = {}
    for key, answer in response.answers.items():
        rows[key] = {name: getattr(answer, name) for name in
                     ("type", "choice", "score", "noul", "confidence", "probabilities", "legend")
                     if hasattr(answer, name)}
    print(json.dumps(rows, ensure_ascii=False, indent=2))

### 0.6 本章离线示例数据

以下数值全部人工构造，专门测试分支；不来自 Jev，也不能用于估计中文准确率或校准情况。正式 live 运行不会使用这些答案。

In [ ]:
URGENCY_OFFLINE = {"urgency": _FakeAnswer("noul", noul=0.96)}
MIXED_OFFLINE = {
    "department": fake_choice({"billing": 0.12, "technical": 0.86, "sales": 0.02}, 0.79),
    "frustration": fake_score({0: 0.08, 1: 0.85, 2: 0.07},
        ["平静地陈述事实", "沮丧但保持礼貌", "非常愤怒或使用强烈措辞"], 0.84),
    "is_urgent": _FakeAnswer("noul", noul=0.96),
}

## 📖 理论根基：一次请求里有什么

状态是客户消息；问题描述你需要的判断；模型字段选择处理请求的模型。
同一段消息可以同时被用于分类、评分和是非判断。返回的是对象，无需从聊天文本中提取 JSON。
模型提供判断，程序仍要检查调用失败、输出结构和业务结果。

本章实际执行两次业务请求，再加准备部分的一次连通性请求。下方 cURL 仅作为等价入口展示，
不会由 Notebook 自动执行；若自己运行它，会增加一次请求。


[官方原文](https://docs.typesafe.ai/api) · [中文参考](https://bald0wang.github.io/jev-docs-zh/api/) · [官方原文](https://docs.typesafe.ai/sdk/python/usage) · [中文参考](https://bald0wang.github.io/jev-docs-zh/sdk/python/usage/)

## 1. 在 Playground 建立直觉

打开 [Playground](https://console.typesafe.ai/playground)，把下一格的中文文本粘贴到 state。
添加 Noul，问题写“这条消息是否表达紧迫性或时间压力？”，然后执行并观察概率。
接着添加部门 Choice 和不满程度 Score，与第 3 节保持同一套定义。

这是手动体验路线；本教程不会替你登录控制台或安装文档内提及的 agent 技能。

### 第一步：官方示例的中文版本

In [ ]:
TICKET = "你好，我尝试连接 Stripe 账户已经三天了，集成一直失败，正在损失销售额。请尽快帮忙。"

### 第二步：只提一个明确问题

In [ ]:
URGENCY_QUESTIONS = {
    "urgency": Noul(instructions="这条消息是否表达紧迫性或时间压力？"),
}

**观察与理解：** Noul 给出的数值是命题成立的概率，不要使用 bool(概率) 来代替阈值判断。

### 第三步：发起请求

In [ ]:
urgency_response = ts.call(TICKET, URGENCY_QUESTIONS, URGENCY_OFFLINE, "Stripe 紧迫性")

### 第四步：读取结果

In [ ]:
show(urgency_response)

**观察与理解：** 官方页面中的示例数字不是你这次调用的标准答案。记录实际数值，比较含义与字段即可。

## 2. 看懂等价 HTTP 请求

请求为 `POST https://api.typesafe.ai/v1/systemone`。认证头读取环境变量，不能把真实值粘贴到教程或输出。
下面先构造不含密钥的请求体。SDK 最终也使用这种请求模型。

构造最小请求体；这里用 Python 字典演示接口结构。

In [ ]:
HTTP_BODY = {
    "model": MODEL,
    "state": TICKET,
    "questions": {
        "urgency": {"type": "noul", "instructions": "这条消息是否表达紧迫性或时间压力？"},
    },
}

显示可以发送的 JSON。认证头不进入输出。

In [ ]:
print(json.dumps(HTTP_BODY, ensure_ascii=False, indent=2))

**观察与理解：** 把该 JSON 保存到 request.json 后，可使用下一段终端命令；它与上面的 SDK 请求是两个可选入口。

```bash
curl --fail-with-body --max-time 30 https://api.typesafe.ai/v1/systemone \
  -H "Authorization: Bearer $TYPESAFE_API_KEY" \
  -H "Content-Type: application/json" \
  --data-binary @request.json
```

`TYPESAFE_API_KEY` 应由你的本地环境管理方式设置。不要把带真实密钥的命令保存到公开文件中。
如果只创建 `.env` 文件而没有加载到启动 Jupyter 的进程，这里仍然读不到变量。

## 3. 一次请求混用三个原语

原理：工单部门、不满程度与紧迫性都能直接从同一条消息独立判断。
三个问题不需要知道彼此的结果。criteria 中的 key 英文、解释中文。

定义 Choice 的标签边界。

In [ ]:
DEPARTMENTS = {
    "billing": "付款、账单或订阅问题",
    "technical": "故障或集成问题",
    "sales": "价格或购买账户的咨询",
}
FRUSTRATION_LEVELS = ["平静地陈述事实", "沮丧但保持礼貌", "非常愤怒或使用强烈措辞"]

定义完整问题。

In [ ]:
MIXED_QUESTIONS = {
    "department": Choice(instructions="哪个团队应处理这条消息的主要诉求？",
                         criteria=DEPARTMENTS),
    "frustration": Score(instructions="客户在措辞中表达了多大程度的不满？",
                         criteria=FRUSTRATION_LEVELS),
    "is_urgent": Noul(instructions="消息是否表达紧迫性或时间压力？"),
}

**观察与理解：** ‘损失销售额’不自动意味着应该路由 sales；分类应看客户正在请求解决什么。

发送三个问题。

In [ ]:
mixed_response = ts.call(TICKET, MIXED_QUESTIONS, MIXED_OFFLINE, "Stripe 三种原语")

查看结构化结果。

In [ ]:
show(mixed_response)

**观察与理解：** 先看实际 department，再看其 confidence；不满程度与问题严重程度是不同的量表。

核对 Score 的含义：从概率重新计算期望。

In [ ]:
frustration = mixed_response.scores["frustration"]
expected_score = sum(int(level) * p for level, p in frustration.probabilities.items())
print({"返回分数": frustration.score, "加权期望": expected_score,
       "绝对差": abs(frustration.score - expected_score)})

**观察与理解：** SDK 中等级通常是整数；原始 JSON 的对象键是字符串。使用 int(level) 兼容两种展示。

### 常见问题

| 现象 | 检查与处理 |
|---|---|
| 找不到 SDK | 确认安装环境与当前内核一致 |
| 缺少密钥 | 配置启动进程的环境变量，重启内核 |
| 401 / 403 | 核对密钥、账户权限；严格 live 停止 |
| 429、连接失败、超时 | 记录错误，核对额度或网络；不要改成假的低 confidence |
| 数值与教程不同 | 记录实际模型与输入，比较语义和字段，不强求逐位相同 |

## 练习与自查

把 TICKET 改为一条平静的价格咨询，再重跑业务部分。先写下预期部门和原因，后记录实际结果。

<details><summary>参考思路：先完成练习再展开</summary>

可以使用‘请问团队版每个月多少钱？我想比较几个方案。’预期更接近 sales；这只是待验证假设，不能在真实调用中断言概率固定。

</details>

## 小结

| 原语 | 本章问题 |
|---|---|
| Choice | 处理部门 |
| Score | 不满程度 |
| Noul | 是否紧迫 |

下一章：[AI 入门](ai_primer_experiments.ipynb)。

离线运行只说明教材代码能执行。正式交付必须实际运行 live，并阅读每条输出；缺失的分支应记为未观察到。

## 本次执行记录

先关闭连接，再生成记录。下面的 JSON 由实际运行计算，批量执行器会据此检查来源。

In [ ]:
if client is not None:
    client.close()

真实探针只演示行为路径；若据其返回挑选样例，这批样例就不适合再当作无偏准确率测试集。延迟也只是本次网络环境中的观测。

In [ ]:
AUDIT = {
    "kind": "jev_execution_audit",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "sdk": version("typesafe-sdk"), "requested_model": MODEL,
    "mode": RUN_MODE, "ping": PING,
    "real_calls": sum(x["source"] == "live" for x in CALL_LOG),
    "offline_calls": sum(x["source"] == "offline" for x in CALL_LOG),
    "cases": CALL_LOG,
    "coverage": globals().get("COVERAGE", {}),
    "validation_status": "live_executed_requires_review" if (
        PING["source"] == "live" and CALL_LOG
        and all(x["source"] == "live" for x in CALL_LOG)
    ) else "offline_only_not_model_evidence",
}
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))

读完输出后，在本仓库 `notebooks/MAINTENANCE.md` 的验收表中记录日期、真实模型、观察到的分支和偏离预期之处。不要把人工演示数值抄进实测记录。